# Anomaly Detection — Isolation Forest Experiment

This notebook walks through the full Machine Learning workflow for the Real-Time Anomaly Detection System:

1. Generate synthetic transaction data
2. Train an Isolation Forest model
3. Evaluate detection performance
4. Visualise anomaly scores
5. Analyse feature importances


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette('husl')
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
print('Libraries loaded ✅')

## 1. Generate Synthetic Data

In [ ]:
N_NORMAL  = 10_000
N_ANOMALY =    500

# ── Normal samples ──────────────────────────────────────────────────────────
normal_amount = rng.uniform(1.0, 500.0, N_NORMAL)
normal_freq   = rng.integers(1, 11, N_NORMAL).astype(float)
normal_age    = rng.integers(30, 3651, N_NORMAL).astype(float)
normal_label  = np.zeros(N_NORMAL)   # 0 = normal

# ── Anomaly samples (mixed types) ───────────────────────────────────────────
n3 = N_ANOMALY // 3
anom_amount = np.concatenate([
    rng.uniform(5000, 50000, n3),                   # high-amount
    rng.uniform(1, 500, n3),                         # normal amount
    rng.uniform(200, 1500, N_ANOMALY - 2*n3),        # new-account
])
anom_freq = np.concatenate([
    rng.integers(1, 11, n3).astype(float),           # normal freq
    rng.integers(50, 201, n3).astype(float),          # high-freq
    rng.integers(1, 11, N_ANOMALY - 2*n3).astype(float),
])
anom_age = np.concatenate([
    rng.integers(30, 3651, n3).astype(float),        # normal age
    rng.integers(30, 3651, n3).astype(float),        # normal age
    rng.integers(0, 3, N_ANOMALY - 2*n3).astype(float),  # new account
])
anom_label = np.ones(N_ANOMALY)   # 1 = anomaly

# ── Combine into DataFrame ──────────────────────────────────────────────────
df = pd.DataFrame({
    'amount':                np.concatenate([normal_amount, anom_amount]),
    'transaction_frequency': np.concatenate([normal_freq,   anom_freq]),
    'account_age':           np.concatenate([normal_age,    anom_age]),
    'label':                 np.concatenate([normal_label,  anom_label]),
})

df['label_str'] = df['label'].map({0: 'normal', 1: 'anomaly'})
print(df.groupby('label_str').describe().round(2))

## 2. Feature Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
features = ['amount', 'transaction_frequency', 'account_age']
titles   = ['Transaction Amount ($)', 'Transaction Frequency (per hr)', 'Account Age (days)']

for ax, feat, title in zip(axes, features, titles):
    for label, color in [('normal', '#00d4ff'), ('anomaly', '#ff4757')]:
        subset = df[df['label_str'] == label][feat]
        ax.hist(subset, bins=40, alpha=0.7, label=label.capitalize(), color=color)
    ax.set_title(title, fontsize=12)
    ax.legend()
    ax.set_xlabel(feat)

plt.suptitle('Feature Distributions: Normal vs Anomaly', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Train Isolation Forest

In [ ]:
X = df[features].values
y_true_binary = df['label'].values   # 0=normal, 1=anomaly

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train on full dataset (unsupervised — no labels used during training)
model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=RANDOM_STATE,
)
model.fit(X_scaled)
print('Isolation Forest trained ✅')

# Predict: sklearn returns +1 (normal) / -1 (anomaly)
y_pred_sklearn = model.predict(X_scaled)
y_pred_binary  = (y_pred_sklearn == -1).astype(int)   # 1=anomaly

# Decision scores (lower = more anomalous)
raw_scores = model.decision_function(X_scaled)
# Normalise to [0,1]: higher = more anomalous
anomaly_scores = 1.0 - (1.0 / (1.0 + np.exp(-raw_scores * 10)))

df['anomaly_score'] = anomaly_scores
df['predicted']     = y_pred_binary

## 4. Model Evaluation

In [ ]:
print('Classification Report')
print('=' * 50)
print(classification_report(
    y_true_binary, y_pred_binary,
    target_names=['Normal', 'Anomaly'],
    digits=3
))

roc = roc_auc_score(y_true_binary, anomaly_scores)
print(f'ROC-AUC Score: {roc:.4f}')

In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(y_true_binary, y_pred_binary)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted Normal', 'Predicted Anomaly'],
    yticklabels=['Actual Normal', 'Actual Anomaly'],
    ax=ax
)
ax.set_title('Confusion Matrix', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Anomaly Score Timeline

In [ ]:
# Sample 300 events to simulate a mini time-series view
sample = df.sample(300, random_state=RANDOM_STATE).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(sample.index, sample['anomaly_score'], color='#00d4ff', linewidth=0.8, alpha=0.7, label='Score')
ax.axhline(0.55, color='#ff4757', linestyle='--', linewidth=1.2, label='Threshold (0.55)')
ax.fill_between(
    sample.index, sample['anomaly_score'], 0.55,
    where=sample['anomaly_score'] >= 0.55,
    color='#ff4757', alpha=0.3, label='Anomaly zone'
)
ax.set_xlabel('Event Index')
ax.set_ylabel('Anomaly Score')
ax.set_title('Anomaly Score Timeline (sample of 300 events)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## 6. 2-D Feature Scatter (Amount vs Frequency)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for lbl, color, marker in [('normal','#00d4ff','o'), ('anomaly','#ff4757','X')]:
    subset = df[df['label_str'] == lbl]
    ax.scatter(
        subset['amount'], subset['transaction_frequency'],
        c=color, marker=marker, s=12, alpha=0.5, label=lbl.capitalize()
    )
ax.set_xlabel('Amount ($)')
ax.set_ylabel('Transaction Frequency (per hr)')
ax.set_title('Feature Space: Amount vs Frequency', fontsize=13)
ax.set_xlim(-500, 55000)
ax.legend()
plt.tight_layout()
plt.show()

print('\nPhase 2 complete — Isolation Forest model successfully trained and evaluated!')